# 后效大盘分布分析 —— 外循环后效

**数据来源**：`ks_ad_antispam.ks_anticheat_small_log_hi`  
**ocpc_action_type**：`EVENT_ORDER_SUBMIT / AD_CID_ROAS / CID_ROAS / CID_EVENT_ORDER_PAID`  
**过滤**：`medium_attribute NOT IN (2,4)`（排除联盟），`visitor_id > 0`，`is_duplicate=false`，`is_retry=false`  
**分析维度**：`visitor_id`  
**分箱方式**：首尾各1个尾部开区间（p5以下 / p95以上），中间8个等距分箱，共10箱  
**指标**：excycle_cost / excycle_order_cnt / excycle_refund_cnt / excycle_refund_rate / excycle_roi

---

## Step 1：配置（修改 P_DATE 后运行）

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

P_DATE = '20260320'

EXCYCLE_OCPC_TYPES = "'EVENT_ORDER_SUBMIT','AD_CID_ROAS','CID_ROAS','CID_EVENT_ORDER_PAID'"

## Step 2：确认 Spark（KML 平台已预注入 spark，无需手动创建）

In [ ]:
print(spark)
print('Spark version:', spark.version)

## Step 3：从 smalllog 按 visitor_id 聚合外循环后效指标

In [ ]:
sql_raw = f"""
SELECT
    visitor_id,
    SUM(CASE WHEN action_type = charge_action_type THEN cost_total ELSE 0 END) / 1000.0                 AS excycle_cost,
    CAST(COUNT(DISTINCT IF(action_type = 'EVENT_ORDER_SUBMIT', llsid, NULL)) AS DOUBLE)                 AS excycle_order_cnt,
    CAST(COUNT(DISTINCT IF(action_type = 'EVENT_PAID_REFUND',  llsid, NULL)) AS DOUBLE)                 AS excycle_refund_cnt,
    SUM(CASE WHEN action_type = 'EVENT_ORDER_SUBMIT' THEN callback_purchase_amount ELSE 0 END) / 1000.0 AS excycle_order_amt
FROM ks_ad_antispam.ks_anticheat_small_log_hi
WHERE p_date = '{P_DATE}'
  AND ocpc_action_type IN ({EXCYCLE_OCPC_TYPES})
  AND is_duplicate = false
  AND is_retry = false
  AND visitor_id > 0
  AND medium_attribute NOT IN (2, 4)
GROUP BY visitor_id
"""

print('正在从 Hive 取数...')
df_raw = spark.sql(sql_raw).toPandas()
print(f'取数完成，visitor_id 数: {len(df_raw):,}')
df_raw.head()

## Step 4：计算合成指标

In [ ]:
df = df_raw.copy()

df['excycle_refund_rate'] = df.apply(
    lambda r: r['excycle_refund_cnt'] / r['excycle_order_cnt'] if r['excycle_order_cnt'] > 0 else None, axis=1
)
df['excycle_roi'] = df.apply(
    lambda r: r['excycle_order_amt'] / r['excycle_cost'] if (r['excycle_cost'] is not None and r['excycle_cost'] > 0) else None, axis=1
)

print(f'总 visitor_id 数: {len(df):,}')
df.describe()

## Step 5：工具函数

In [ ]:
def tail_equal_width_distribution(df, value_col, n_mid=8):
    s = df[value_col].dropna()
    s = s[s > 0]
    if s.empty:
        print(f'  [WARN] {value_col}: 无有效数据')
        return pd.DataFrame()

    p5  = s.quantile(0.05)
    p95 = s.quantile(0.95)

    mid_edges = np.linspace(p5, p95, n_mid + 1)
    edges = np.unique(np.concatenate([[0], mid_edges, [np.inf]]))

    labels = []
    for i in range(len(edges) - 1):
        lo, hi = edges[i], edges[i + 1]
        lo_str = '0' if lo == 0 else f'{lo:.4f}'
        hi_str = '+inf' if hi == np.inf else f'{hi:.4f}'
        labels.append(f'({lo_str}, {hi_str}]')

    bins = pd.cut(s, bins=edges, labels=labels, include_lowest=True, right=True)
    result = (
        s.groupby(bins, observed=True)
        .agg(cnt='count', min_val='min', max_val='max', avg_val='mean', total_val='sum')
        .reset_index()
    )
    result.columns = ['分箱区间', '用户数cnt', '最小值', '最大值', '均值', '总量']
    result['占比%'] = (result['用户数cnt'] / result['用户数cnt'].sum() * 100).round(2)
    result['p5边界']  = round(p5,  4)
    result['p95边界'] = round(p95, 4)
    return result


def show_distribution(result, metric_name, save_csv=True):
    print(f"\n{'='*70}")
    print(f'  {metric_name}  (visitor_id维度，尾部开区间+中间等距，共10箱)')
    print(f"{'='*70}")
    if result.empty:
        return
    print(result.to_string(index=False))
    if save_csv:
        fname = f'dist_excycle_{metric_name}.csv'
        result.to_csv(fname, index=False)
        print(f'  => 已保存: {fname}')

## Step 6：各指标分布分析

### 6.1 外循环消耗（excycle_cost）

In [ ]:
show_distribution(tail_equal_width_distribution(df, 'excycle_cost'), 'excycle_cost')

### 6.2 下单数（excycle_order_cnt）

In [ ]:
show_distribution(tail_equal_width_distribution(df, 'excycle_order_cnt'), 'excycle_order_cnt')

### 6.3 退单数（excycle_refund_cnt）

In [ ]:
show_distribution(tail_equal_width_distribution(df, 'excycle_refund_cnt'), 'excycle_refund_cnt')

### 6.4 退单率（excycle_refund_rate）

In [ ]:
show_distribution(tail_equal_width_distribution(df, 'excycle_refund_rate'), 'excycle_refund_rate')

### 6.5 外循环ROI（excycle_roi）

In [ ]:
show_distribution(tail_equal_width_distribution(df, 'excycle_roi'), 'excycle_roi')

## Step 7：汇总总览（p50 / p90 / p99）

In [ ]:
metric_cols = [
    'excycle_cost', 'excycle_order_cnt', 'excycle_refund_cnt',
    'excycle_refund_rate', 'excycle_roi'
]

summary = []
for col in metric_cols:
    s = df[col].dropna()
    s = s[s > 0]
    if s.empty:
        summary.append({'口径': col, '有效UV数': 0})
        continue
    summary.append({
        '口径': col,
        '有效UV数': len(s),
        '均值': round(s.mean(), 4),
        'p50': round(s.quantile(0.5), 4),
        'p90': round(s.quantile(0.9), 4),
        'p99': round(s.quantile(0.99), 4),
    })

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))
summary_df.to_csv('excycle_summary.csv', index=False)
print('\n=> 已保存: excycle_summary.csv')